# 11 性能优化与工程化

数据量从 1 万行涨到 1000 万行时，同一段 pandas 代码可能从 0.1 秒变成 10 分钟。
这一节不讲玄学，只讲**真正有效的六件事**：

1. 把 `dtype` 选对（能省 50%~90% 内存）；
2. 用向量化代替 `apply` / `iterrows`；
3. 字符串操作走 `.str` 访问器；
4. 只读需要的列、尽早过滤、尽早聚合；
5. 用方法链 + `pipe` 把管线写成可读的流水线；
6. 避开 Copy-on-Write 的坑。

> 环境：`llm_sft`，从 `python/pandas` 目录启动。下面所有耗时都在本机测得，你的机器上数值会不同，但**相对差距**是一致的。

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)

DATA = Path("data")
if not DATA.exists():
    DATA = Path("python/pandas/data")
OUT = DATA.parent / "output"
OUT.mkdir(exist_ok=True)

orders = pd.read_csv(DATA / "sales_orders.csv", parse_dates=["order_date"])

# 把 60 行放大到 2 万行，模拟真实项目
rng = np.random.default_rng(0)
big = pd.concat([orders] * 334, ignore_index=True)
big["order_id"] = "X" + big.index.astype(str)
big["amount"] = (big["amount"] + rng.normal(0, 10, len(big))).round(2)
print("行数:", len(big))
big.dtypes

行数: 20040


order_id                 str
order_date    datetime64[us]
city                     str
category                 str
channel                  str
unit_price           float64
qty                    int64
amount               float64
sales_rep                str
dtype: object

In [2]:
def timeit(func, repeat: int = 2) -> float:
    """跑 repeat 次，返回最快的一次（单位：秒）。"""
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter()
        func()
        best = min(best, time.perf_counter() - start)
    return best


print("计时代码准备就绪")

计时代码准备就绪


## 1. dtype：内存的第一杀手

| 优化 | 效果 | 注意 |
| --- | --- | --- |
| 文本列 → `category` | 重复值多时省 90%+ | 取值种类多（>50%）时反而更慢 |
| `float64` → `float32` | 省一半 | 精度降低，金额别随便降 |
| `int64` → `int8/int16` | 省很多 | 注意取值范围 |
| 只用需要的列 | 线性省内存 | `usecols` 在读取阶段就生效 |

In [3]:
def mem_mb(frame: pd.DataFrame) -> float:
    return frame.memory_usage(deep=True).sum() / 1024**2


before = mem_mb(big)

optimized = big.copy()
for col in ["city", "category", "channel", "sales_rep"]:
    optimized[col] = optimized[col].astype("category")
optimized["qty"] = optimized["qty"].astype("int8")
optimized["unit_price"] = optimized["unit_price"].astype("float32")

after = mem_mb(optimized)
print(f"优化前: {before:6.1f} MB")
print(f"优化后: {after:6.1f} MB  （省了 {(1 - after / before) * 100:.0f}%）")

pd.DataFrame({
    "dtype": big.dtypes.astype(str),
    "每列MB": (big.memory_usage(deep=True) / 1024**2).round(3),
    "优化后MB": (optimized.memory_usage(deep=True) / 1024**2).round(3),
})

优化前:    1.9 MB
优化后:    0.7 MB  （省了 62%）


,dtype,每列MB,优化后MB
Index,NaN,0.000,0.000
amount,float64,0.153,0.153
category,str,0.268,0.019
channel,str,0.268,0.019
city,str,0.268,0.019
order_date,datetime64[us],0.153,0.153
order_id,str,0.257,0.257
qty,int64,0.153,0.019
sales_rep,str,0.266,0.019
unit_price,float64,0.153,0.076


> **别过度优化**：`order_id` 有 2 万个不同取值，转成 `category` 只会更慢更大。
> 经验法则：**基数（不同取值数）/ 行数 < 20%** 才考虑 `category`。

In [4]:
print("order_id 基数:", big["order_id"].nunique(), "/", len(big))
print("city 基数    :", big["city"].nunique(), "/", len(big))

order_id 基数: 20040 / 20040
city 基数    : 6 / 20040


## 2. 向量化 > `apply` > `iterrows`

差距通常有 10~100 倍。原因是：向量化走的是 C 层循环，`iterrows` 每行都要造一个 Series 对象。

In [5]:
t_vec = timeit(lambda: big["amount"] * 1.1)
t_apply = timeit(lambda: big["amount"].apply(lambda v: v * 1.1))
t_iter = timeit(lambda: [row.amount * 1.1 for row in big.itertuples()])

print(f"向量化        : {t_vec * 1000:8.1f} ms")
print(f"Series.apply  : {t_apply * 1000:8.1f} ms  （慢 {t_apply / t_vec:.0f} 倍）")
print(f"itertuples    : {t_iter * 1000:8.1f} ms  （慢 {t_iter / t_vec:.0f} 倍）")

向量化        :      0.1 ms


Series.apply  :      1.8 ms  （慢 25 倍）
itertuples    :     78.9 ms  （慢 1103 倍）


In [6]:
# 条件赋值：np.select 一次算完，别用 apply 逐行 if/elif
conditions = [big["amount"] >= 3000, big["amount"] >= 1000, big["amount"] >= 500]
choices = [0.85, 0.90, 0.95]

t_select = timeit(lambda: np.select(conditions, choices, default=1.0))
t_rowwise = timeit(lambda: big.apply(lambda r: 0.85 if r.amount >= 3000 else (0.9 if r.amount >= 1000 else 0.95 if r.amount >= 500 else 1.0), axis=1))

print(f"np.select           : {t_select * 1000:8.1f} ms")
print(f"DataFrame.apply(axis=1): {t_rowwise * 1000:8.1f} ms  （慢 {t_rowwise / t_select:.0f} 倍）")

np.select           :      0.2 ms
DataFrame.apply(axis=1):    222.5 ms  （慢 1451 倍）


## 3. 字符串操作：用 `.str`，不要 `apply(re.search)`

In [7]:
t_str = timeit(lambda: big["city"].str.startswith("北"))
t_re = timeit(lambda: big["city"].apply(lambda s: s.startswith("北")))

print(f".str.startswith : {t_str * 1000:8.1f} ms")
print(f"apply + startswith: {t_re * 1000:8.1f} ms  （慢 {t_re / t_str:.1f} 倍）")

.str.startswith :      0.2 ms
apply + startswith:      4.4 ms  （慢 19.0 倍）


## 4. 读取与过滤：越早越小

一条原则：**能在读取阶段丢掉的，就不要读进内存；能在聚合前过滤的，就不要先合并。**

In [8]:
big.to_csv(OUT / "perf_big.csv", index=False)

t_all = timeit(lambda: pd.read_csv(OUT / "perf_big.csv"))
t_cols = timeit(lambda: pd.read_csv(OUT / "perf_big.csv", usecols=["amount"]))
t_filtered = timeit(lambda: pd.read_csv(OUT / "perf_big.csv", usecols=["amount", "city"]).query("amount > 2000"))

print(f"读全部          : {t_all * 1000:8.1f} ms")
print(f"只读 1 列       : {t_cols * 1000:8.1f} ms")
print(f"只读 2 列 + 过滤: {t_filtered * 1000:8.1f} ms")

读全部          :     21.7 ms
只读 1 列       :      9.9 ms
只读 2 列 + 过滤:     11.9 ms


In [9]:
# 先聚合再 merge：把 2 万行压成 6 行，再去和城市维表拼
cities = pd.DataFrame({"city": ["北京", "上海", "广州", "深圳", "杭州", "成都"], "region": ["华北", "华东", "华南", "华南", "华东", "西南"]})

t_merge_first = timeit(lambda: big.merge(cities, on="city").groupby("region")["amount"].sum())
t_agg_first = timeit(lambda: big.groupby("city", as_index=False)["amount"].sum().merge(cities, on="city"))

print(f"先 merge 再聚合: {t_merge_first * 1000:8.1f} ms")
print(f"先聚合再 merge : {t_agg_first * 1000:8.1f} ms")

先 merge 再聚合:      4.4 ms
先聚合再 merge :      2.6 ms


## 5. 方法链 + `pipe`：让管线可读

数据清洗天然是"一串步骤"。与其写十个中间变量，不如写成流水线：

```python
result = (df.pipe(step1).pipe(step2).pipe(step3))
```

好处：每一步都能单独测试、单独替换，出问题时一眼看出是哪一步。

In [10]:
def drop_bad_rows(df: pd.DataFrame) -> pd.DataFrame:
    return df[(df["amount"] > 0) & (df["qty"] > 0)]


def add_discount(df: pd.DataFrame) -> pd.DataFrame:
    conditions = [df["amount"] >= 3000, df["amount"] >= 1000]
    return df.assign(discount=np.select(conditions, [0.85, 0.9], default=1.0))


def add_payable(df: pd.DataFrame) -> pd.DataFrame:
    return df.assign(payable=(df["amount"] * df["discount"]).round(2))


def summarize_by_city(df: pd.DataFrame) -> pd.DataFrame:
    return df.groupby("city", as_index=False).agg(订单数=("order_id", "count"), 实付=("payable", "sum"))


pipeline_result = (
    big.pipe(drop_bad_rows)
    .pipe(add_discount)
    .pipe(add_payable)
    .pipe(summarize_by_city)
    .sort_values("实付", ascending=False)
)
pipeline_result

,city,订单数,实付
4,杭州,4342,4738102.87
3,成都,1670,4259348.01
5,深圳,3991,3313107.06
2,广州,4008,3124513.27
1,北京,3003,2952585.63
0,上海,2004,2892227.80


## 6. Copy-on-Write 与常见性能坑

| 坑 | 后果 | 正确做法 |
| --- | --- | --- |
| `df[cond]["col"] = v` | 静默无效 | `df.loc[cond, "col"] = v` |
| 循环里 `df = df.append(row)` | O(n²)，越跑越慢 | 收集到 list，最后 `pd.concat` |
| `df.copy()` 一大坨 | 内存翻倍 | 能链式就链式，确实要改再 copy |
| `groupby.apply` 返回 DataFrame | 慢且结果形状不可控 | 用 `agg` / `transform` |
| `pd.concat` 在循环里 | 每次全量复制 | 先 append 到 list，再一次 concat |
| 对 object 列做 `str` | 回退到 Python 循环 | 先 `astype("string")` |

In [11]:
# 反例：循环里 concat（先别跑，太慢）；正解：list + 一次 concat
parts = [big.iloc[i : i + 1000] for i in range(0, 5000, 1000)]
fast = pd.concat(parts, ignore_index=True)
print("list + concat 结果:", fast.shape)

rows = [{"city": c, "n": len(g)} for c, g in big.groupby("city")]
print(pd.DataFrame(rows))

list + concat 结果: (5000, 9)
   city     n
0  上海  2338
1  北京  3006
2  广州  4008
3  成都  1670
4  杭州  4342
5  深圳  4676


In [12]:
# 复制一整张表是有代价的：2 万行还算便宜，2000 万行就是几个 GB
t_copy = timeit(lambda: big.copy())
print(f"copy 一次: {t_copy * 1000:.1f} ms")

copy 一次: 0.1 ms


## 7. 小练习

下面这段代码在一张 200 万行的表上要跑十几秒。请把它改写成向量化版本（提示：`np.select` + `where`）：

```python
for i in range(len(df)):
    if df.at[i, "amount"] > 3000:
        df.at[i, "tier"] = "A"
    elif df.at[i, "amount"] > 1000:
        df.at[i, "tier"] = "B"
    else:
        df.at[i, "tier"] = "C"
```

然后在 2 万行的 `big` 上对比两种写法的耗时。

In [13]:
def tier_loop(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for i in range(len(df)):
        if df.at[i, "amount"] > 3000:
            df.at[i, "tier"] = "A"
        elif df.at[i, "amount"] > 1000:
            df.at[i, "tier"] = "B"
        else:
            df.at[i, "tier"] = "C"
    return df


def tier_vectorized(df: pd.DataFrame) -> pd.DataFrame:
    return df.assign(tier=np.select([df["amount"] > 3000, df["amount"] > 1000], ["A", "B"], default="C"))


t_loop = timeit(lambda: tier_loop(big.head(5000)), repeat=1)
t_vec = timeit(lambda: tier_vectorized(big))
print(f"for 循环 5000 行 : {t_loop * 1000:8.1f} ms")
print(f"向量化 2 万行    : {t_vec * 1000:8.1f} ms")

a = tier_loop(big.head(200))["tier"].astype(str)
b = tier_vectorized(big.head(200))["tier"].astype(str)
print("两者结果一致:", a.equals(b))

for 循环 5000 行 :    265.3 ms
向量化 2 万行    :      1.7 ms
两者结果一致: True


## 8. 小结

**性能清单（按收益排序）**

1. 只读需要的列（`usecols`），尽早过滤（`query`），尽早聚合。
2. 向量化 / `.str` / `np.where` / `np.select` 替换 `apply`、`iterrows`、`for`。
3. 文本列基数低就转 `category`，浮点别轻易降精度。
4. 循环里不要 `concat` / `append`，先攒 list 再一次合并。
5. 大文件用 `chunksize` 或 Parquet。
6. `groupby` 优先 `agg` / `transform`；`apply` 是最后手段。

**工程化清单**

- 每一步清洗写成小函数，用 `pipe` 串起来；
- 每个函数只做一件事，输入输出都是 DataFrame；
- 函数里不要用全局变量，参数显式传入；
- 中间结果存 Parquet，随时能从任意一步重跑。